# The Whole Cycle, On Your Own

**Notebook 5 of 5.** Nothing new is taught. You build a forest end to end and compare it with
the best single tree, tuning on the out-of-bag score so the test set is touched once, at the
end.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

## 1. The data

`data/concrete.csv`: 1,030 mixes, seven ingredients, an age in days, and
`compressive_strength` in MPa.

> **Exercise:**
>
> Load it, separate the target, and hold out 20% with `random_state=42` — the same split as
> every other notebook here, so your scores are comparable.

<details><summary>
Click here for a hint...
</summary>

`train_test_split(X, y, test_size=0.2, random_state=42)`. No `stratify`: the target is a
number.

</details>

In [ ]:
df = pd.read_csv("../data/concrete.csv")

X = df.drop(columns="compressive_strength")
y = df["compressive_strength"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"{len(X_train)} training mixes, {len(X_test)} test mixes")

## 2. The number to beat

The decision trees repository tuned a single tree by cross-validation and settled on no depth
limit with `min_samples_leaf=2`.

> **Exercise:**
>
> Fit that tree and report its test MAE, RMSE and R². Everything below has to beat it.

<details><summary>
Click here for a hint...
</summary>

`DecisionTreeRegressor(min_samples_leaf=2, random_state=42)`, then the three metrics on
`X_test`.

</details>

In [ ]:
def report(name, model):
    """Print the test MAE, RMSE and R-squared of a fitted regressor.

    Args:
        name: label for the printed row.
        model: a fitted scikit-learn regressor.
    """
    predictions = model.predict(X_test)
    print(f"{name:22s} MAE {mean_absolute_error(y_test, predictions):5.2f}"
          f"   RMSE {root_mean_squared_error(y_test, predictions):5.2f}"
          f"   R² {r2_score(y_test, predictions):6.3f}")


tree = DecisionTreeRegressor(min_samples_leaf=2, random_state=42).fit(X_train, y_train)
report("tuned single tree", tree)

<details><summary>
Click here to compare your answer...
</summary>

**MAE 4.66, RMSE 6.95, R² 0.812.** That is the best a single tree managed on this data.

</details>

## 3. A forest, out of the box

> **Exercise:**
>
> Fit a `RandomForestRegressor` with 200 trees, `oob_score=True` and nothing else changed.
> Report its test metrics and its out-of-bag R².

<details><summary>
Click here for a hint...
</summary>

`oob_score_` holds the out-of-bag R² once the forest is fitted.

</details>

In [ ]:
default_forest = RandomForestRegressor(
    n_estimators=200, oob_score=True, random_state=42, n_jobs=-1
)
default_forest.fit(X_train, y_train)

report("forest, defaults", default_forest)
print(f"{'':22s} out-of-bag R² {default_forest.oob_score_:.3f}")

<details><summary>
Click here to compare your answer...
</summary>

**MAE 3.80, RMSE 5.58, R² 0.879**, against the tree's 4.66, 6.95 and 0.812: nearly a full MPa
off the average miss, with no setting chosen.

Note this forest is really bagging — `RandomForestRegressor` defaults to `max_features=1.0`.

</details>

## 4. Tune it without the test set

> **Exercise:**
>
> Fit a forest for every combination of `max_features` in `[1.0, 0.5, "sqrt"]` and
> `min_samples_leaf` in `[1, 2, 5]`, with `oob_score=True`. Print the out-of-bag R² of each and
> keep the best combination.

<details><summary>
Click here for a hint...
</summary>

Two nested loops, nine fits, a couple of seconds each. Track the best as you go, or collect
the results and take the maximum at the end.

</details>

In [ ]:
results = []
for max_features in [1.0, 0.5, "sqrt"]:
    for min_samples_leaf in [1, 2, 5]:
        candidate = RandomForestRegressor(
            n_estimators=200, max_features=max_features, min_samples_leaf=min_samples_leaf,
            oob_score=True, random_state=42, n_jobs=-1,
        )
        candidate.fit(X_train, y_train)
        results.append((candidate.oob_score_, max_features, min_samples_leaf, candidate))
        print(f"max_features={str(max_features):5s}  min_samples_leaf={min_samples_leaf}  "
              f"out-of-bag R² {candidate.oob_score_:.4f}")

# the best row by out-of-bag score: no test data was consulted to get here
best_score, best_features, best_leaf, forest = max(results, key=lambda row: row[0])
print(f"\nbest: max_features={best_features}, min_samples_leaf={best_leaf}")

<details><summary>
Click here to compare your answer...
</summary>

**`max_features=0.5` with `min_samples_leaf=1`**, at an out-of-bag R² of 0.9164.

Two patterns worth reading. Larger leaves always hurt here, unlike the census data of notebook
04 where they were the one thing that mattered. And `"sqrt"` was the worst column at every
leaf size: eight features cannot spare five at every split.

</details>

## 5. The test set, once

> **Exercise:**
>
> Report the tuned forest's test metrics beside the single tree's.

<details><summary>
Click here for a hint...
</summary>

The loop above already kept the fitted model, so there is nothing to refit.

</details>

In [ ]:
report("tuned single tree", tree)
report("forest, defaults", default_forest)
report("tuned forest", forest)

<details><summary>
Click here to compare your answer...
</summary>

| | MAE | RMSE | R² |
| :-- | --: | --: | --: |
| tuned single tree | 4.66 | 6.95 | 0.812 |
| forest, defaults | 3.80 | 5.58 | 0.879 |
| **tuned forest** | **3.71** | **5.33** | **0.890** |

The average miss falls from 4.66 MPa to 3.71, a fifth off, and most of that came from using a
forest at all — tuning added the last tenth.

The out-of-bag R² was 0.9164 against a test R² of 0.890: optimistic again, by the same margin
as in notebook 04, and right about the ranking, which is what it was asked for.

</details>

## 6. Where it still goes wrong

> **Exercise:**
>
> Split the test mixes into four groups by true strength with `pd.qcut`, and report the
> forest's mean absolute error in each. The single tree's were **2.43, 4.65, 3.82 and 7.70**
> from weakest to strongest.

<details><summary>
Click here for a hint...
</summary>

Build a frame of the true value and the absolute error, then group by
`pd.qcut(frame["true"], 4)`.

</details>

In [ ]:
errors = pd.DataFrame({
    "true": y_test,
    "abs_error": (y_test - forest.predict(X_test)).abs(),
})

quartiles = pd.qcut(errors["true"], 4)
print(errors.groupby(quartiles, observed=True)["abs_error"].mean().round(2).to_string())

<details><summary>
Click here to compare your answer...
</summary>

**2.76, 3.36, 2.66 and 6.02**, against the tree's 2.43, 4.65, 3.82 and 7.70.

The forest wins in the three upper groups and **loses in the weakest**, 2.76 against 2.43:
averaging pulls predictions toward the middle, and the weakest mixes are ones a single tree
could reach by memorizing them.

The strongest quartile stays the hardest for both, and for the same reason — a leaf predicts a
mean, and extremes sit at the edge of whatever landed in it.

</details>

## 7. What it leaned on

> **Exercise:**
>
> Run `permutation_importance` on the tuned forest with the test set, `n_repeats=10`,
> `random_state=42` and `scoring="r2"`, and rank the features. The single tree gave **age
> 0.816** and **cement 0.736**.

<details><summary>
Click here for a hint...
</summary>

`.importances_mean` into a `Series` indexed by `X.columns`.

</details>

In [ ]:
importance = permutation_importance(
    forest, X_test, y_test, n_repeats=10, random_state=42, scoring="r2"
)

pd.Series(importance.importances_mean, index=X.columns).sort_values(ascending=False).round(3)

<details><summary>
Click here to compare your answer...
</summary>

**Age 0.792, cement 0.378, water 0.195**, the rest well behind.

Age barely moved from the tree's 0.816, but **cement halved**, 0.736 to 0.378. Nothing about
the concrete changed. The model did: 200 trees, each forbidden some features at each split,
found other routes to the same answer, so scrambling cement costs this model less.

An importance number describes **a model**, not the data.

</details>

## Summary

- A forest beat the best single tree **out of the box**: MAE 3.80 against 4.66.
- **Out-of-bag tuning** cost nine fits and no test data: MAE **3.71**, R² **0.890**.
- The forest wins in three quartiles of four and is **worse on the weakest mixes**, where
  averaging pulls toward the middle.
- **Importance describes the model**: cement halved between tree and forest, on identical data.

Every tree here grew independently of the others and was averaged at the end. The next step is
to grow them **in sequence**, each fixing what the ones before got wrong.